In [ ]:

import optuna
import pandas as pd
from common import (
    DATASETS,
    METHODS,
    METRIC_TO_LABEL,
    get_color,
    get_dataset_label,
    get_method_label,
)
from matplotlib import pyplot as plt

optuna_storage = optuna.storages.RDBStorage()

In [ ]:
def update_cache():
    records = []
    for dataset in DATASETS:
        for method in METHODS:
            study_name = f"bayescl/hp/{dataset}/{method}"
            n = 0
            try:
                study = optuna.load_study(study_name=study_name, storage=optuna_storage)
                df = study.trials_dataframe()
                df = df[df.state == "COMPLETE"]
                n = len(df)
                for trial in df.itertuples():
                    records.append(
                        {
                            "dataset": dataset,
                            "method": method,
                            "accuracy_seen_avg": trial.values_0,
                            "ece_seen_avg": trial.values_1,
                        }
                    )
            except KeyError:
                pass
            print(f"{study_name}, {n}")

    df = pd.DataFrame(records)
    df.to_csv("dataframe/hpsearch.csv", index=False)


update_cache()

In [ ]:
df = pd.read_csv("dataframe/hpsearch.csv")
df.groupby(["dataset", "method"]).aggregate(
    {
        "accuracy_seen_avg": ["max"],
        "ece_seen_avg": ["min", "count"],
    }
).to_csv("dataframe/hpsearch_summary.csv")

In [ ]:
from common import get_marker

LIMIT = 10
width = 3
fig, axies = plt.subplots(figsize=(width * 3, width), ncols=3)

for i, (dataset) in enumerate(DATASETS):
    ax = axies[i]
    for method in METHODS:
        df_filtered = df[(df.method == method) & (df.dataset == dataset)][:LIMIT]

        ax.scatter(
            df_filtered["accuracy_seen_avg"],
            df_filtered["ece_seen_avg"],
            color=get_color(method),
            marker=get_marker(method),
        )
        ax.set_title(get_dataset_label(dataset))

        # Make figure a square
        ax.set_box_aspect(1)

    axies[1].set_xlabel(METRIC_TO_LABEL.get("accuracy_seen_avg"))
    axies[0].set_ylabel(METRIC_TO_LABEL.get("ece_seen_avg"))

# Add legend to the right of the figure
handles = [
    plt.Line2D(
        [0],
        [0],
        marker=get_marker(method),
        color=get_color(method),
        markersize=10,
        linestyle="None",
    )
    for method in METHODS
]
labels = [get_method_label(method) for method in METHODS]
fig.legend(handles, labels, loc="center right", frameon=False)
plt.tight_layout(rect=[0, 0, 0.86, 1])
plt.savefig("plots/hpsearch_plot.pdf")
plt.savefig("plots/hpsearch_plot.png")